# Lab 00: Distributions, Divergences, and Numerics

**Tier 1 lab.** Every cell here ran during the build, and every claim is checked with an
`assert` statement, which means the notebook fails loudly if any claim stops being true. There
is no training in this lab and no GPU requirement; it runs unchanged inside a GPU container
(arm64 included) and on any laptop. The code auto-detects CUDA, and every assertion is
device-independent by construction, meaning it passes or fails the same way on any hardware.

This is the prerequisite refresher for the whole course. The question it answers is the one the
README puts first: **what is actually being minimised?** Every Tier 2 recipe later in the
course, whether it is TRL's `DistillationTrainer`, training from cached top-k logits (storing
only each position's k highest-scoring vocabulary entries, covered in Unit 04), or on-policy
scoring (having the teacher grade text the student itself generated, covered in Unit 07),
bottoms out in the twenty lines of log-space arithmetic this lab verifies. Log-space arithmetic
means working with log-probabilities instead of probabilities, and the sections below show why
that choice is forced rather than stylistic. If any section here feels obvious, run it anyway;
the assertions cost seconds, and two of them (§7 on number formats, §9 on sampled estimators)
encode traps that bite working engineers on exactly the hardware this course targets.

In [1]:
import sys, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import kl_divergence, gjsd, tvd, masked_mean

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device}")
if device == "cuda":
    print(f"  {torch.cuda.get_device_name(0)} | capability sm_{''.join(map(str, torch.cuda.get_device_capability(0)))}")

def rand_dist(*shape, peak=1.0, generator=None):
    '''Random categorical distribution via softmax of scaled Gaussian logits.'''
    return F.softmax(peak * torch.randn(*shape, generator=generator), dim=-1)

torch 2.13.0+cpu | device: cpu


## 1. Softmax is a claim about relative evidence, and the naive form overflows

A language model head emits a vector of logits $z \in \mathbb{R}^V$ (one raw score per
vocabulary entry, $V$ of them), and the probability distribution over next tokens is

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}.$$

Two properties matter for everything downstream. First, softmax is **shift invariant**:
$\mathrm{softmax}(z + c) = \mathrm{softmax}(z)$ for any scalar $c$, because the factor $e^c$
appears in both the numerator and the denominator and cancels. So only logit *differences*
carry information; the absolute values mean nothing. Second, the textbook formula is **not
computable** as written. Every number format has a largest value it can store, and any
calculation whose result exceeds it produces the special value `inf`; that event is called
overflow. The fp32 format tops out around 3.4e38, and since 88.7 is the natural log of that
maximum, `exp` of any logit above roughly 88 overflows fp32. The fp16 format tops out at
65504, whose natural log is about 11.1, so `exp` of a logit above roughly 11 overflows fp16.
Real logits from real checkpoints sit in the 10 to 40 range, and the moment you divide them by
a small temperature (temperature scaling divides every logit by a constant $T$, which Lab 01
does constantly; $T = 0.25$ multiplies every logit by 4) you are in overflow territory.

Shift invariance is also the fix: subtract the maximum logit from every logit before
exponentiating. The distribution is unchanged, and the largest exponent is now exactly 0, so
nothing overflows. Every library does this internally, which is precisely why you should see
it fail once by hand: if you ever write your own low-level implementation of this math, or
read a cache of stored teacher scores written by someone who saved `exp(z)` "for speed",
nothing will warn you that the protection is gone.

In [2]:
def naive_softmax(z):
    e = z.exp()
    return e / e.sum(-1, keepdim=True)

def stable_softmax(z):
    zs = z - z.max(-1, keepdim=True).values
    e = zs.exp()
    return e / e.sum(-1, keepdim=True)

# Realistic failure: logits around 30, temperature 0.25 -> effective logits around 120.
z = torch.tensor([31.0, 29.5, 27.0, 12.0]) / 0.25

p_naive  = naive_softmax(z)
p_stable = stable_softmax(z)
p_torch  = F.softmax(z, dim=-1)

print(f"naive : {p_naive.tolist()}")
print(f"stable: {[f'{v:.6f}' for v in p_stable.tolist()]}")

assert torch.isnan(p_naive).any(), "expected the naive form to overflow to NaN"
assert torch.allclose(p_stable, p_torch, atol=1e-7), "stable form must match F.softmax"

# Shift invariance, asserted rather than stated.
z2 = torch.randn(3, 8)
assert torch.allclose(F.softmax(z2, -1), F.softmax(z2 + 123.4, -1), atol=1e-6)
print("\nnaive overflows, stable matches F.softmax, shift invariance holds")

naive : [nan, nan, nan, 0.0]
stable: ['0.997527', '0.002473', '0.000000', '0.000000']

naive overflows, stable matches F.softmax, shift invariance holds


## 2. `logsumexp` is the primitive everything else is built on

The log-partition function $\mathrm{LSE}(z) = \log \sum_j e^{z_j}$ is the log of the softmax
denominator, and it gives log-softmax directly:

$$\log p_i = z_i - \mathrm{LSE}(z),$$

and the same max-subtraction trick from §1 makes it exact:
$\mathrm{LSE}(z) = m + \log \sum_j e^{z_j - m}$ with $m = \max_j z_j$, which is the same
quantity algebraically but never exponentiates anything larger than 0.

The practical rule this section verifies: **never take `log` of a probability you got from
`softmax`; get log-probabilities directly from `log_softmax`.** Here is why the two paths
differ. A computer stores numbers in a format (fp32) with a smallest representable positive
value, roughly 1e-45. Any calculation whose true answer is smaller than that gets stored as
exactly 0.0; this is called underflow. Path one: `softmax` computes actual probabilities as
stored numbers. A very unlikely token's true probability might be exp(-110), which is about
1e-48. That is below the floor, so it is stored as 0.0, and then `log(0.0)` returns
negative infinity. Path two: `log_softmax` computes the same quantity through the identity
log p_i = z_i - logsumexp(z), which only ever subtracts ordinary-sized numbers (logits like
-110 and 5) from each other. The tiny probability never gets computed or stored at any
point, so nothing underflows, and the result is the correct answer, -110, as a perfectly
normal float. Same math, different order of operations, and only one order survives the
number format. The last thing to know is what the bug looks like from the outside: if that
-inf reaches a KL computation, the loss becomes inf or NaN, and on a training dashboard
that looks like "training became unstable". You would waste hours tuning learning rates
when the actual problem is one wrong function call.

In [3]:
def my_logsumexp(z):
    m = z.max(-1, keepdim=True).values
    return (m + (z - m).exp().sum(-1, keepdim=True).log()).squeeze(-1)

z = torch.randn(4, 100, 512) * 10
assert torch.allclose(my_logsumexp(z), torch.logsumexp(z, -1), atol=1e-5)

# log∘softmax underflows where log_softmax does not.
z = torch.tensor([0.0, -110.0])         # a token the model has ruled out
composed = F.softmax(z, -1).log()
direct   = F.log_softmax(z, -1)

print(f"log(softmax(z)) = {composed.tolist()}")
print(f"log_softmax(z)  = {direct.tolist()}")

assert torch.isinf(composed[1]), "expected underflow -> log(0) = -inf"
assert torch.isfinite(direct[1]) and abs(direct[1].item() - (-110.0)) < 1e-3
print("\ncomposed form hit -inf; log_softmax returned the exact value -110")

log(softmax(z)) = [0.0, -inf]
log_softmax(z)  = [0.0, -110.0]

composed form hit -inf; log_softmax returned the exact value -110


## 3. Entropy, cross-entropy, KL: one identity to remember

Three quantities, all measured in nats. A nat is the unit of information you get when you use
natural logarithms; using base-2 logs instead would give bits. For distributions $p$ (the
reference you are matching) and $q$ (your model): the entropy $H(p) = -\sum_i p_i \log p_i$
measures how spread out $p$ is, the cross-entropy $H(p, q) = -\sum_i p_i \log q_i$ is the
familiar training loss, and the KL divergence
$\mathrm{KL}(p\,\|\,q) = \sum_i p_i (\log p_i - \log q_i)$ measures how far $q$ is from $p$.
They obey one identity:

$$H(p, q) = H(p) + \mathrm{KL}(p \,\|\, q).$$

Cross-entropy is the total price you pay; entropy is the irreducible part, the price a perfect
model would still pay because $p$ itself is uncertain; KL is the remainder, the part that is
*your model's fault*. Three consequences worth internalising:

- **Hard-label training is KD with a one-hot teacher.** A one-hot distribution puts
  probability 1 on a single token and 0 everywhere else. When $p$ is one-hot, $H(p)=0$
  because there is no uncertainty to pay for, so the identity says cross-entropy *is* forward
  KL. Distillation with a soft teacher is not a different objective from ordinary hard-label
  training; it is the same objective with a better reference distribution.
- **Perplexity is $e^{H(p,q)}$** (per token). Perplexity is the standard language-modelling
  metric, read as "the model is as uncertain as if it were choosing uniformly among this many
  options". Because it is just the exponential of cross-entropy, and cross-entropy is entropy
  plus KL, a perplexity gap between student and teacher is a KL statement in disguise.
- **Gibbs' inequality**: $\mathrm{KL} \ge 0$ always, with equality exactly when $p = q$. This
  is why a loss of zero is achievable in principle, and why any *negative* KL you ever see in
  a training log is a bug, full stop.

In [4]:
g = torch.Generator().manual_seed(1)
p = rand_dist(6, 32, generator=g)
q = rand_dist(6, 32, generator=g)

H_p  = -(p * p.log()).sum(-1)
H_pq = -(p * q.log()).sum(-1)
kl   = (p * (p.log() - q.log())).sum(-1)

assert torch.allclose(H_pq, H_p + kl, atol=1e-6), "H(p,q) = H(p) + KL(p||q)"
assert (kl >= -1e-7).all(), "Gibbs: KL is non-negative"
assert (p * (p.log() - p.log())).sum(-1).abs().max() < 1e-9, "KL(p||p) = 0"

# One-hot teacher: forward KL == cross-entropy == the ordinary hard-label loss.
onehot = F.one_hot(torch.tensor([3]), num_classes=32).float()
z_student = torch.randn(1, 32)
kl_onehot = (onehot * (onehot.clamp_min(1e-30).log() - F.log_softmax(z_student, -1))).sum(-1)
ce_hard   = F.cross_entropy(z_student, torch.tensor([3]))
assert torch.allclose(kl_onehot.squeeze(), ce_hard, atol=1e-6)

print(f"mean H(p) = {H_p.mean():.4f} nats, mean KL = {kl.mean():.4f} nats, "
      f"mean perplexity vs q = {H_pq.mean().exp():.2f}")
print("H(p,q) = H(p) + KL verified; one-hot teacher reduces KD to cross-entropy")

mean H(p) = 3.0096 nats, mean KL = 0.9519 nats, mean perplexity vs q = 52.53
H(p,q) = H(p) + KL verified; one-hot teacher reduces KD to cross-entropy


## 4. KL is not a distance

A distance, in the mathematical sense, must be symmetric (the distance from a to b equals the
distance from b to a) and must satisfy the triangle inequality (going from a to c directly is
never longer than going a to b and then b to c). KL fails both: it is asymmetric, with
$\mathrm{KL}(p\,\|\,q) \ne \mathrm{KL}(q\,\|\,p)$ in general, and it violates the triangle
inequality. Neither failure is a technicality. The asymmetry is the entire content of the
choice between forward and reverse KL (Lab 01 §4 shows what each one does to a bimodal
teacher, one whose distribution has two separated peaks), and the triangle violation means
you cannot reason "the student is close to an intermediate model, the intermediate is close
to the teacher, therefore the student is close to the teacher". That is exactly the reasoning
a multi-stage distillation pipeline (Unit 09's prune-then-distill, or setups that insert a
mid-sized teacher assistant between teacher and student) quietly tempts you into.

One genuinely useful metric fact survives: $\sqrt{\mathrm{JSD}}$, the square root of the
Jensen-Shannon divergence defined in §5, **is** a metric, symmetric and triangle-obeying.
When you need a number that behaves like a distance (clustering checkpoints, plotting how far
a model has drifted over training), use that, not raw KL.

In [5]:
g = torch.Generator().manual_seed(2)

# Asymmetry: peaked p vs broad q.
zp = torch.tensor([[8.0, 0.0, 0.0, 0.0]]); zq = torch.zeros(1, 4)
m = torch.ones(1, 1, dtype=torch.bool)
fwd = kl_divergence(zq[None], zp[None], m, direction="forward")   # KL(p||q)
rev = kl_divergence(zq[None], zp[None], m, direction="reverse")   # KL(q||p)
print(f"KL(peaked||broad) = {fwd:.4f}   KL(broad||peaked) = {rev:.4f}")
assert abs(fwd - rev) > 1.0, "KL should be grossly asymmetric here"

# Triangle inequality: search random triples for KL(p||r) > KL(p||q) + KL(q||r).
def kl_pq(p, q): return (p * (p.log() - q.log())).sum(-1)
violations = 0
for _ in range(2000):
    p, q, r = (rand_dist(4, peak=3.0, generator=g) for _ in range(3))
    if (kl_pq(p, r) > kl_pq(p, q) + kl_pq(q, r)).item():
        violations += 1
print(f"triangle-inequality violations: {violations}/2000 random triples")
assert violations > 0, "KL must violate the triangle inequality somewhere"

# sqrt(JSD) is a metric: no violations, ever.
def jsd(p, q):
    mm = 0.5 * (p + q)
    return 0.5 * kl_pq(p, mm) + 0.5 * kl_pq(q, mm)
bad = 0
for _ in range(2000):
    p, q, r = (rand_dist(4, peak=3.0, generator=g) for _ in range(3))
    if (jsd(p, r).sqrt() > jsd(p, q).sqrt() + jsd(q, r).sqrt() + 1e-7).item():
        bad += 1
assert bad == 0, "sqrt(JSD) is a metric; found a triangle violation"
print("sqrt(JSD) survived 2000 triples with zero violations")

KL(peaked||broad) = 1.3772   KL(broad||peaked) = 4.6147


triangle-inequality violations: 460/2000 random triples


sqrt(JSD) survived 2000 triples with zero violations


## 5. The f-divergence family: one generator, many objectives

Every divergence this course uses is an instance of one formula. Pick a convex function $f$
with $f(1)=0$, called the generator, and define

$$D_f(p \,\|\, q) = \sum_i q_i \, f\!\left(\frac{p_i}{q_i}\right).$$

The intuition: the ratio $p_i/q_i$ equals 1 wherever the two distributions agree, and
$f(1)=0$ means agreement costs nothing; the generator decides how heavily each kind of
disagreement gets charged. Different choices of $f$ produce every divergence in this course:

| divergence | generator $f(t)$ |
|---|---|
| forward KL $(p\|q)$ | $t \log t$ |
| reverse KL $(q\|p)$ | $-\log t$ |
| total variation | $\tfrac{1}{2}\lvert t - 1 \rvert$ |
| Jensen–Shannon | $\tfrac{1}{2}\!\left[t \log t - (1+t)\log\tfrac{1+t}{2}\right]$ |

This is not taxonomy for its own sake. The 2026 on-policy distillation literature organises
methods by their f-divergence, and TRL's `beta` knob (Lab 01 §5) is a walk along this family.
When a new paper announces a new objective, your first move is to find its generator. Two
behaviours then follow mechanically instead of experimentally: whether the objective is mode
seeking or mode covering (whether a student too small to match the whole teacher commits to
one high-probability region of the teacher's distribution or spreads itself across all of
them; Lab 01 §4 demonstrates both), and whether it is bounded (whether a single bad token can
contribute an arbitrarily large loss; §6 below).

The cell verifies the table by computing each divergence *twice*, once from the generator and
once from the direct formulas in `kd_core`, and asserting they agree.

In [6]:
def f_div(p, q, f):
    t = p / q
    return (q * f(t)).sum(-1)

g = torch.Generator().manual_seed(3)
zp, zq = torch.randn(2, 7, 48, generator=g), torch.randn(2, 7, 48, generator=g)
p, q = F.softmax(zp, -1), F.softmax(zq, -1)
m = torch.ones(2, 7, dtype=torch.bool)

pairs = [
    ("forward KL", lambda t: t * t.log(),
     kl_divergence(zq, zp, m, direction="forward", scale_by_T2=False)),
    ("reverse KL", lambda t: -t.log(),
     kl_divergence(zq, zp, m, direction="reverse", scale_by_T2=False)),
    ("TVD", lambda t: 0.5 * (t - 1).abs(),
     tvd(zq, zp, m)),
    ("JSD", lambda t: 0.5 * (t * t.log() - (1 + t) * ((1 + t) / 2).log()),
     gjsd(zq, zp, m, beta=0.5)),
]
for name, f, direct in pairs:
    via_gen = masked_mean(f_div(p, q, f), m)
    assert torch.allclose(via_gen, direct, atol=1e-5), f"{name}: generator != direct"
    print(f"{name:>10}: generator {via_gen:.6f} == kd_core {float(direct):.6f}")
print("\nfour divergences, one formula — table verified")

forward KL: generator 1.004068 == kd_core 1.004068
reverse KL: generator 1.027349 == kd_core 1.027349
       TVD: generator 0.528443 == kd_core 0.528442
       JSD: generator 0.205109 == kd_core 0.205109

four divergences, one formula — table verified


## 6. Bounds you can lean on

Three inequalities carry real operational weight:

- $\mathrm{JSD} \le \log 2$ and $\mathrm{TVD} \le 1$, always, including on **disjoint
  support**, the extreme case where every token that $p$ considers possible has probability
  zero under $q$ and vice versa. On disjoint support KL is infinite, because it contains a
  $\log(p_i/q_i)$ term with $q_i = 0$ in the denominator. This is the precise sense in which
  the bounded divergences are a safety valve (Lab 01 §8 shows the consequence for gradients):
  no single token can contribute an unbounded term to the loss.
- **Pinsker's inequality**: $\mathrm{TVD} \le \sqrt{\mathrm{KL}/2}$. TVD, the total variation
  distance, is the largest difference in probability that the two distributions assign to any
  event, so read Pinsker as a promise: drive KL down and every event probability the student
  assigns converges to the teacher's at a known rate. Plug in the numbers: a KL of 0.02 nats
  gives $\sqrt{0.02/2} = 0.1$, so TVD is capped at 10%, whatever the vocabulary size.

The disjoint-support case is not exotic. Early in training, a student can assign essentially
zero mass to a token the teacher is confident about, and §7's demonstration of number formats
shows fp16 manufacturing exact zeros out of merely-small probabilities, which turns "nearly
disjoint" into "actually disjoint".

In [7]:
g = torch.Generator().manual_seed(4)
log2 = math.log(2.0)

worst_jsd, worst_pinsker_slack = 0.0, float("inf")
for _ in range(3000):
    peak = float(torch.empty(1).uniform_(0.5, 6.0, generator=g))
    p, q = rand_dist(16, peak=peak, generator=g), rand_dist(16, peak=peak, generator=g)
    kl_v  = (p * (p.log() - q.log())).sum(-1)
    tvd_v = 0.5 * (p - q).abs().sum(-1)
    mm = 0.5 * (p + q)
    jsd_v = 0.5 * (p * (p.log() - mm.log())).sum(-1) + 0.5 * (q * (q.log() - mm.log())).sum(-1)
    assert jsd_v <= log2 + 1e-6 and tvd_v <= 1.0 + 1e-6
    assert tvd_v <= (kl_v / 2).sqrt() + 1e-6, "Pinsker violated"
    worst_jsd = max(worst_jsd, float(jsd_v))
    worst_pinsker_slack = min(worst_pinsker_slack, float((kl_v / 2).sqrt() - tvd_v))

print(f"3000 random pairs: max JSD {worst_jsd:.4f} <= log 2 = {log2:.4f}; "
      f"Pinsker min slack {worst_pinsker_slack:.4f} >= 0")

# Disjoint support: KL blows up, the bounded pair saturates politely.
p = torch.tensor([0.5, 0.5, 0.0, 0.0]); q = torch.tensor([0.0, 0.0, 0.5, 0.5])
pe, qe = p.clamp_min(1e-12), q.clamp_min(1e-12)     # epsilon stands in for exact zeros
kl_dj  = (pe * (pe.log() - qe.log())).sum()
mm = 0.5 * (pe + qe)
jsd_dj = 0.5 * (pe * (pe.log() - mm.log())).sum() + 0.5 * (qe * (qe.log() - mm.log())).sum()
tvd_dj = 0.5 * (p - q).abs().sum()
print(f"disjoint support: KL = {kl_dj:.1f} nats (-> inf as eps -> 0), "
      f"JSD = {jsd_dj:.4f} (= log 2), TVD = {tvd_dj:.1f} (= 1)")
assert kl_dj > 10 and abs(jsd_dj - log2) < 1e-3 and abs(tvd_dj - 1.0) < 1e-6

3000 random pairs: max JSD 0.6931 <= log 2 = 0.6931; Pinsker min slack 0.0191 >= 0
disjoint support: KL = 26.9 nats (-> inf as eps -> 0), JSD = 0.6931 (= log 2), TVD = 1.0 (= 1)


## 7. Floating point: where the objective actually lives

Tier 2 runs will hold model weights in **bf16**, a 16-bit floating-point format that modern
accelerators are built to run fast. Storing each parameter in 2 bytes instead of fp32's 4 is
what makes a 32B-parameter teacher fit in a 128 GB unified-memory budget: 32B parameters
times 2 bytes is 64 GB, leaving room for the student and everything else. So it matters
exactly where in the loss computation you are allowed to be in 16 bits, and where you are not.

A quick anatomy lesson, because the failure modes below depend on it. A floating-point format
splits its bits between an exponent, which sets the *range* of magnitudes the format can
represent, and a mantissa, which sets the *precision*, meaning how many significant digits
survive. Machine eps in the table is the gap between 1.0 and the next representable number,
the standard measure of that precision.

| format | exponent | mantissa | max finite | machine eps |
|---|---|---|---|---|
| fp32 | 8 | 23 | ~3.4e38 | 1.2e-7 |
| bf16 | 8 | 7 | ~3.4e38 | 7.8e-3 |
| fp16 | 5 | 10 | 65504 | 9.8e-4 |

The two 16-bit formats split the bit budget differently and therefore fail differently.
**fp16** spent its bits on mantissa, so it has decent precision but a tiny range: it
overflows at 65504 (a temperature-scaled logit can pass that, as §1 computed), and any
probability below about 6e-8, the smallest positive value fp16 can represent, gets stored as
exact zero. That manufactures the disjoint-support case of §6 out of perfectly healthy
distributions. **bf16** spent its bits on exponent, so it has fp32's full range but only a
7-bit mantissa, roughly 2 to 3 significant decimal digits: nothing overflows, and instead
every third digit of your loss is noise. On a training dashboard that noise does not look
like an error; it looks like a plateau, or like a run that gives different numbers each time
you repeat it.

The rule, and it is the one Hugging Face and TRL implement internally: **run the model in
bf16, compute the divergence in fp32.** The upcast costs one fp32 copy of the logit tensor
(shape [batch, sequence length, vocabulary]) at the loss site and nothing else. The cell
measures what each policy actually does to a KL value.

In [8]:
g = torch.Generator().manual_seed(5)
zp = 12 * torch.randn(64, 256, generator=g)     # peaked, realistic post-head scale
zq = 12 * torch.randn(64, 256, generator=g)

def kl_in(dtype):
    lp = F.log_softmax(zp.to(dtype), -1).float()
    lq = F.log_softmax(zq.to(dtype), -1).float()
    return (lp.exp() * (lp - lq)).sum(-1).mean()

ref = float(kl_in(torch.float64))
for dt in (torch.float32, torch.bfloat16, torch.float16):
    v = float(kl_in(dt))
    print(f"log_softmax in {str(dt):>15}: KL = {v:12.6f}   rel err {abs(v-ref)/ref:.2e}")

err32 = abs(float(kl_in(torch.float32)) - ref) / ref
err16 = abs(float(kl_in(torch.bfloat16)) - ref) / ref
assert err32 < 1e-5, "fp32 loss math should be clean"
assert err16 > 10 * err32, "bf16 loss math should be visibly worse than fp32"

# fp16's flush-to-zero manufactures -inf out of a merely-unlikely token.
p16 = F.softmax((torch.tensor([0.0, -18.0]) / 0.5), -1).half()   # ~2e-16 < fp16 tiny
assert p16[1] == 0.0, "fp16 flushed a small probability to exact zero"
print(f"\nfp16 turned probability ~2e-16 into exact {float(p16[1])} -> log gives -inf")
print("rule verified: model in bf16, divergence in fp32")

log_softmax in   torch.float32: KL =    34.186119   rel err 1.12e-07
log_softmax in  torch.bfloat16: KL =    34.247696   rel err 1.80e-03
log_softmax in   torch.float16: KL =    34.187923   rel err 5.29e-05

fp16 turned probability ~2e-16 into exact 0.0 -> log gives -inf
rule verified: model in bf16, divergence in fp32


## 8. The gradient you should be able to derive on a whiteboard

For a fixed reference $p$ and student logits $z$ with $q = \mathrm{softmax}(z)$:

$$\frac{\partial}{\partial z_i} \, \mathrm{KL}(p \,\|\, q) = q_i - p_i.$$

The gradient of the entire objective is *the residual between the two distributions*: at each
vocabulary entry, the update pushes the student's probability toward the teacher's by an
amount equal to their current gap, and nothing else. Everything Lab 01 establishes about
temperature is a corollary of this formula: soften both sides with a temperature and the
residual $q_i - p_i$ shrinks by a factor of $1/T$, which is where the $T^2$ compensation
factor comes from (Lab 01 §2 walks through the full accounting). If you remember one piece of
calculus from this course, make it this one. It is also the fastest way to sanity-check any
custom loss: let `autograd` differentiate the scalar and compare the result against
$q - p$.

In [9]:
g = torch.Generator().manual_seed(6)
p = rand_dist(3, 40, generator=g)
z = torch.randn(3, 40, generator=g, requires_grad=True)

loss = (p * (p.log() - F.log_softmax(z, -1))).sum()
loss.backward()

q = F.softmax(z, -1)
assert torch.allclose(z.grad, q - p, atol=1e-6), "grad of KL(p||softmax(z)) must be q - p"
print(f"max |autograd - (q - p)| = {(z.grad - (q - p)).abs().max():.2e}")
print("the whiteboard gradient matches autograd")

max |autograd - (q - p)| = 2.24e-08
the whiteboard gradient matches autograd


## 9. Estimating a KL you cannot afford to compute

Every divergence so far summed over the full vocabulary. That is fine when you have dense
logits, meaning the teacher's complete score vector at every position, which off-policy
cached-logit training (Unit 04: compute the teacher's outputs once, store them, replay them
during training) arranges to have. On-policy distillation (Unit 07) does not: there the
student *samples* tokens, generating text one random draw at a time, the teacher scores only
the token that was actually sampled, and the full-vocabulary sum is replaced by a Monte Carlo
estimate, an average over random draws that approaches the true value only as draws
accumulate. Which estimator you average is a real decision with a failure mode attached.

Sample $x \sim q$ and write the ratio $r(x) = p(x)/q(x)$. Three estimators of
$\mathrm{KL}(q\|p)$, named k1, k2, k3 after Schulman's note on approximating KL:

$$k_1 = -\log r, \qquad k_2 = \tfrac{1}{2}(\log r)^2, \qquad k_3 = (r - 1) - \log r.$$

Two properties matter for each estimator: bias (whether its average converges to the true KL)
and variance (how noisy the individual samples are). $k_1$ is unbiased but noisy: individual
samples are *negative* whenever the teacher likes the sampled token more than the student
does, even though the true KL is positive, so the running estimate wobbles around the truth.
$k_2$ is low-variance but biased. $k_3$ is unbiased (because $\mathbb{E}_q[r] = 1$, so the
extra $r - 1$ term averages to zero while cancelling much of $k_1$'s noise) and every sample
is non-negative.

But the variance comparison between $k_1$ and $k_3$ is **regime-dependent**, and the cell
below demonstrates both regimes because the build originally asserted "k3 always wins" and
the data refused. When $q \approx p$, which is the late-training regime, $k_3$'s variance is
far below $k_1$'s. When the student is *far* from the teacher, the ratio $r$ explodes on rare
tokens (a token with teacher probability 0.1 and student probability 0.0001 gives $r$ of
1000), and $k_3$'s variance blows past $k_1$'s by orders of magnitude. That is precisely the
**off-policy cold start** problem Unit 07 treats: sampled-ratio estimators are trustworthy
exactly when the two models already roughly agree, and a freshly initialised student is the
case where they do not. The failure modes the 2026 on-policy literature reports for
sampled-token estimators are this variance story surfacing at scale.

In [10]:
g = torch.Generator().manual_seed(7)
V, N = 500, 200_000

def run_regime(tag, p, q):
    true_kl = float((q * (q.log() - p.log())).sum())
    x = torch.multinomial(q, N, replacement=True, generator=g)
    log_r = (p.log() - q.log())[x]
    r = log_r.exp()
    ks = {"k1": -log_r, "k2": 0.5 * log_r ** 2, "k3": (r - 1) - log_r}
    print(f"{tag}: true KL(q||p) = {true_kl:.5f} nats")
    print(f"{'':>4} {'mean':>10} {'std':>12} {'bias':>10}")
    for name, k in ks.items():
        print(f"{name:>4} {k.mean():>10.5f} {k.std():>12.5f} {k.mean() - true_kl:>+10.5f}")
    print()
    return true_kl, ks

# Regime A — late training: student close to the teacher.
z = torch.randn(V, generator=g) * 2.0
p_close = F.softmax(z, -1)
q_close = F.softmax(z + 0.3 * torch.randn(V, generator=g), -1)
kl_a, ka = run_regime("A (q ~ p)", p_close, q_close)

# Regime B — cold start: student far from the teacher.
p_far = rand_dist(V, peak=2.0, generator=g)
q_far = rand_dist(V, peak=2.0, generator=g)
kl_b, kb = run_regime("B (q far from p)", p_far, q_far)

for kl_t, ks in ((kl_a, ka), (kl_b, kb)):
    se1 = float(ks["k1"].std()) / math.sqrt(N); se3 = float(ks["k3"].std()) / math.sqrt(N)
    assert abs(float(ks["k1"].mean()) - kl_t) < 5 * se1, "k1 is unbiased"
    assert abs(float(ks["k3"].mean()) - kl_t) < 5 * se3, "k3 is unbiased"
    assert (ks["k3"] >= -1e-6).all(), "every k3 sample is non-negative"
    assert (ks["k1"] < 0).any(), "k1 goes negative on individual samples"
assert float(ka["k3"].std()) < float(ka["k1"].std()), "close regime: k3 beats k1"
assert float(kb["k3"].std()) > float(kb["k1"].std()), "far regime: k3's ratio term explodes"
print("k1/k3 unbiased in both regimes; k3 wins on variance only when q ~ p — verified")

A (q ~ p): true KL(q||p) = 0.05328 nats
           mean          std       bias
  k1    0.05233      0.33106   -0.00095
  k2    0.05617      0.07118   +0.00289
  k3    0.05323      0.06122   -0.00006

B (q far from p): true KL(q||p) = 4.06400 nats
           mean          std       bias
  k1    4.06940      2.21122   +0.00539
  k2   10.72473      7.62477   +6.66073
  k3    3.96930     23.85423   -0.09470

k1/k3 unbiased in both regimes; k3 wins on variance only when q ~ p — verified


## Exercises

1. **Break bf16 on purpose.** Rerun §7 with the logit scale at 1, 12, and 40. At which scale
   does bf16 loss math first disagree with fp32 in the second significant digit? Relate the
   answer to the mantissa widths in the table: bf16 keeps roughly 2 to 3 significant digits,
   so the disagreement should appear once the computation demands more than that.
2. **Pinsker in reverse.** Pinsker's inequality bounds TVD by KL, but no bound exists in the
   other direction. Construct a sequence of distribution pairs with TVD → 0 and KL → ∞
   (hint: move a shrinking amount of probability mass onto a token where $q$ is
   astronomically small; the shrinking mass drives TVD down while the enormous log-ratio
   drives KL up). This is the precise reason a small measured TVD does not certify a small
   KL, and it previews the tail-mass discussion in Lab 02.
3. **Your own generator.** The χ² divergence has generator $f(t) = (t-1)^2$. Add it to the §5
   harness, verify it against a direct formula, and check whether it is bounded. Would you
   train against it? Why not? (Consider what squaring the ratio does to the §9 variance
   story, where $r$ appearing only to the first power was already the problem.)
4. **k3 for the forward direction.** Derive the k3-style estimator for
   $\mathrm{KL}(p \| q)$ from samples of $q$. You will need importance weighting, meaning
   each sample's contribution is reweighted by a probability ratio to correct for drawing
   from $q$ rather than $p$. Measure its variance against the §9 setup and explain why
   on-policy *forward*-KL distillation is harder than reverse (this is Unit 07's `beta`
   discussion arriving early).